In [61]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", message=".*urllib3 v2 only supports OpenSSL.*")


In [100]:
from qiskit_aer import AerSimulator


backend = AerSimulator()
print(f"backend initialised successfully: {backend}")

backend initialised successfully: AerSimulator('aer_simulator')


In [101]:
from qiskit import QuantumCircuit
import random
identity = QuantumCircuit(2)
identity.cx(0,1)
identity_gate = identity.to_gate()
identity_gate.name = "identity"

complement = QuantumCircuit(2)
complement.x(0)
complement.cx(0,1)
complement.x(0)
complement_gate = complement.to_gate()
complement_gate.name = "complement"

always_1 = QuantumCircuit(2)
always_1.x(1)
always_1_gate = always_1.to_gate()
always_1_gate.name = "always_1"

always_0 = QuantumCircuit(2)
always_0_gate = always_0.to_gate()
always_0_gate.name = "always_0"

oracle_gate = random.choice([identity_gate,complement_gate,always_1_gate,always_0_gate])
print(f"{oracle_gate.name}")

oracle_test = QuantumCircuit(2,1)
oracle_test.h(0)
oracle_test.x(1)
oracle_test.h(1)
oracle_test.barrier()
oracle_test.append(oracle_gate,[0,1])
oracle_test.barrier()
oracle_test.h(0)
oracle_test.measure(0,0)
oracle_test.draw(output='text')

identity


┌───┐      ░ ┌───────────┐ ░ ┌───┐┌─┐
q_0: ┤ H ├──────░─┤0          ├─░─┤ H ├┤M├
     ├───┤┌───┐ ░ │  identity │ ░ └───┘└╥┘
q_1: ┤ X ├┤ H ├─░─┤1          ├─░───────╫─
     └───┘└───┘ ░ └───────────┘ ░       ║ 
c: 1/═══════════════════════════════════╩═
                                        0

In [102]:
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
pm = generate_preset_pass_manager(optimization_level=1, backend= backend)
isa_circuit = pm.run(oracle_test)

Level,Name,Primary Focus,Best For

0. No Optimization,"Pure, raw translation",Debugging & Hardware Research
1. Light Optimization,Basic gate clean-up & fast routing,Local Simulations (Our choice today!)
2. Heavy Optimization,Noise-aware routing & gate merging,Standard physical QPU execution
3. Even Heavier Optimization,Deep matrix synthesis & maximum compression,High-stakes production hardware runs

In [115]:
from qiskit.primitives import StatevectorSampler
sampler = StatevectorSampler()
job = sampler.run([isa_circuit],shots=1)
result=job.result()
data_packet = result[0]
data_packet.data.c.get_counts()

{'1': 1}